# Part 1.3 (a): Predicting Late Deliveries on the Olist Marketplace

**Dataset:** Brazilian E-Commerce Public Dataset by Olist,
<https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce>

**Question:** at the moment a parcel is handed to the carrier, can we tell whether it will miss the
delivery date the customer was promised, and what drives that?

Framing the prediction at **carrier handover** is what makes the model useful, because a warning
that arrives after the parcel is already late is worth nothing. That rule decides which columns are
allowed: anything recorded after the parcel reaches the customer is excluded, including the review
score.

Two algorithms are compared on identical features: `XGBClassifier` (gradient boosting) and `SVC`.


## Step 1: Imports

In [1]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score, classification_report, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
print("libraries imported")

libraries imported


## Step 2: Paths and configuration

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd().parent
OLIST_DIR = ROOT / "olist"
OUTPUT_DIR = ROOT / "analysis" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fitting an RBF support vector machine costs roughly O(n^2) in training rows,
# so it is fitted on a subsample of this size. See Step 15.
SVM_TRAIN_CAP = 15_000

print(f"reading from {OLIST_DIR}")

reading from /Users/ashish/Desktop/RMIT/Case studies/olist


## Step 3: Load the nine source files

The category translation file carries a byte order mark
that corrupts its first column name, and the reviews file has newlines inside quoted comments so a raw line count overstates its size.

In [3]:
def read(name):
    return pd.read_csv(OLIST_DIR / f"olist_{name}_dataset.csv")

orders = read("orders")
order_items = read("order_items")
order_payments = read("order_payments")
order_reviews = read("order_reviews")
customers = read("customers")
sellers = read("sellers")
products = read("products")
geolocation = read("geolocation")
# utf-8-sig strips the byte order mark; without it the merge below silently fails.
category_translation = pd.read_csv(
    OLIST_DIR / "product_category_name_translation.csv", encoding="utf-8-sig"
)

print(f"orders {len(orders):,} | items {len(order_items):,} | geolocation {len(geolocation):,}")

orders 99,441 | items 112,650 | geolocation 1,000,163


## Step 4: Filter to delivered orders and define the target

An order is **late** when it reached the customer after the estimated delivery date shown at
checkout. That is the promise the customer was actually given, which is what matters commercially.
Only delivered orders have an outcome to learn from.

In [4]:
date_cols = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

df = orders[orders["order_status"] == "delivered"].copy()
# Both timestamps are essential: one defines the target, the other the prediction moment.
df = df.dropna(subset=["order_delivered_customer_date", "order_delivered_carrier_date"])

df["late"] = (
    df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]
).astype(int)

print(f"delivered orders with both timestamps : {len(df):,}")
print(f"late rate                             : {df['late'].mean():.2%}")

delivered orders with both timestamps : 96,469
late rate                             : 8.11%


## Step 5: Why accuracy is the wrong metric here

This single fact drives every metric decision in the notebook. At roughly 8% positives, a model
that predicts "never late" scores about 92% accuracy while catching precisely zero late
deliveries. Accuracy is therefore reported only to show it is misleading, and average precision is
used as the headline metric because it measures performance on the rare class alone.

In [5]:
naive_accuracy = 1 - df["late"].mean()
print(f"accuracy of a 'never late' model : {naive_accuracy:.2%}")
print(f"late deliveries it would catch   : 0 of {df['late'].sum():,}")

accuracy of a 'never late' model : 91.89%
late deliveries it would catch   : 0 of 7,825


## Step 6: Timing and calendar features

In [6]:
HOUR, DAY = 3600.0, 86400.0

df["approval_lag_hours"] = (
    df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / HOUR
df["seller_processing_days"] = (
    df["order_delivered_carrier_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / DAY
df["promised_window_days"] = (
    df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / DAY
# Share of the promised window already spent before the carrier takes over.
df["window_used_at_handover"] = df["seller_processing_days"] / df["promised_window_days"]

df["purchase_month"] = df["order_purchase_timestamp"].dt.month
df["purchase_weekday"] = df["order_purchase_timestamp"].dt.dayofweek
df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour
df["purchase_year"] = df["order_purchase_timestamp"].dt.year  # only for the Step 18 holdout

print("4 timing features and 4 calendar features created")

4 timing features and 4 calendar features created


## Step 7: Remove impossible timestamps

A small number of orders record the carrier handover as happening before the customer placed the
order. They are dropped rather than clipped, since clipping would invent a value never observed.

In [7]:
bad_timing = df["seller_processing_days"] < 0
print(f"handover before purchase : {bad_timing.sum():,} ({bad_timing.mean():.3%})")
df = df[~bad_timing].copy()
print(f"rows remaining           : {len(df):,}")

handover before purchase : 165 (0.171%)
rows remaining           : 96,304


## Step 8: Aggregate the item table before merging

`order_items` holds **one row per item**, not per order. Merging it straight onto `orders` would
duplicate every multi-item order and silently inflate the dataset, which is the easiest mistake to
make with this data. It is aggregated to order level first, and each order is attributed to the
seller of its most expensive item, being the seller most likely to drive the dispatch timeline.

In [8]:
items = order_items.merge(products, on="product_id", how="left")
items = items.merge(category_translation, on="product_category_name", how="left")
items["product_volume_cm3"] = (
    items["product_length_cm"] * items["product_height_cm"] * items["product_width_cm"])

item_agg = items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    n_sellers=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    max_product_weight_g=("product_weight_g", "max"),
    total_product_volume_cm3=("product_volume_cm3", "sum"),
    n_photos=("product_photos_qty", "mean"),
)
# Sorting by price means "first" gives the most expensive item.
primary_item = (items.sort_values("price", ascending=False).groupby("order_id")
                .agg(seller_id=("seller_id", "first"),
                     product_category=("product_category_name_english", "first")))
item_agg = item_agg.join(primary_item).reset_index()

print(f"{len(order_items):,} item rows -> {len(item_agg):,} orders")

112,650 item rows -> 98,666 orders


## Step 9: Aggregate the payment table

In [9]:
payment_agg = order_payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    max_installments=("payment_installments", "max"),
    n_payments=("payment_sequential", "count"),
)
# The payment method covering the largest share of the order value.
dominant_payment = (order_payments.sort_values("payment_value", ascending=False)
                    .groupby("order_id")["payment_type"].first().rename("payment_type"))
payment_agg = payment_agg.join(dominant_payment).reset_index()
print(f"{len(order_payments):,} payment rows -> {len(payment_agg):,} orders")

103,886 payment rows -> 99,440 orders


## Step 10: Merge the aggregates, guarding against duplication

In [10]:
rows_before = len(df)
df = df.merge(item_agg, on="order_id", how="inner")
# Fails loudly if the aggregation logic ever reintroduces duplication.
assert len(df) <= rows_before, "order_items merge duplicated orders"
df = df.merge(payment_agg, on="order_id", how="left")
print(f"{rows_before:,} orders before merge -> {len(df):,} after. No duplication.")

96,304 orders before merge -> 96,304 after. No duplication.


## Step 11: Distance between seller and customer

A handful of geolocation rows sit outside Brazil, as far away as Europe and Asia, and produced
delivery distances over 8,000 km before being filtered out. That is roughly twice the width of the
country.

In [11]:
BRAZIL_LAT, BRAZIL_LNG = (-33.75, 5.27), (-73.99, -34.79)
outside = ~(geolocation["geolocation_lat"].between(*BRAZIL_LAT)
            & geolocation["geolocation_lng"].between(*BRAZIL_LNG))
print(f"coordinates outside Brazil: {outside.sum():,} ({outside.mean():.4%})")

centroids = (geolocation[~outside]
             .groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
             .mean().reset_index())

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * radius * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

df = df.merge(customers[["customer_id", "customer_zip_code_prefix", "customer_state"]],
              on="customer_id", how="left")
df = df.merge(sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]],
              on="seller_id", how="left")
for side in ["customer", "seller"]:
    df = df.merge(centroids.rename(columns={
        "geolocation_zip_code_prefix": f"{side}_zip_code_prefix",
        "geolocation_lat": f"{side}_lat", "geolocation_lng": f"{side}_lng"}),
        on=f"{side}_zip_code_prefix", how="left")

df["distance_km"] = haversine_km(df["seller_lat"], df["seller_lng"],
                                 df["customer_lat"], df["customer_lng"])
df["is_interstate"] = (df["customer_state"] != df["seller_state"]).astype(int)

print(f"median distance {df['distance_km'].median():,.0f} km | "
      f"max {df['distance_km'].max():,.0f} km | interstate {df['is_interstate'].mean():.1%}")

coordinates outside Brazil: 42 (0.0042%)


median distance 433 km | max 3,399 km | interstate 64.0%


## Step 12: Collapse the long-tailed categories

In [12]:
def top_n(series, n, other="other"):
    keep = series.value_counts().nlargest(n).index
    return series.where(series.isin(keep), other)

df["product_category"] = top_n(df["product_category"].fillna("unknown"), 15)
df["customer_state"] = top_n(df["customer_state"], 12)
df["seller_state"] = top_n(df["seller_state"], 12)
df["payment_type"] = df["payment_type"].fillna("unknown")
print(f"{df['product_category'].nunique()} categories, {df['customer_state'].nunique()} states kept")

16 categories, 13 states kept


## Step 13: The leakage guard

The review score is attached here to show why it must **not** be a feature.
Late orders score far worse than on-time orders, and that strong relationship is exactly what would flatter the model while making it unusable, because the review does not exist yet at the moment the prediction is needed. Using it would be predicting the past from the future.

It is kept as a descriptive result, since the gap quantifies the commercial cost of lateness.

In [13]:
review_scores = order_reviews.groupby("order_id")["review_score"].mean()
df = df.merge(review_scores, on="order_id", how="left")

summary = df.groupby("late")["review_score"].agg(["mean", "count"])
summary.index = ["on time", "late"]
print(summary.round(2).to_string())
print(f"\nlate orders score {summary.loc['on time','mean'] - summary.loc['late','mean']:.2f} "
      f"stars lower, but this is a consequence of lateness, not a predictor of it")

         mean  count
on time  4.29  88002
late     2.57   7657

late orders score 1.73 stars lower, but this is a consequence of lateness, not a predictor of it


## Step 14: Feature matrix and stratified split

In [14]:
NUMERIC_FEATURES = [
    "seller_processing_days", "approval_lag_hours", "promised_window_days",
    "window_used_at_handover", "distance_km", "is_interstate",
    "total_freight", "total_price", "payment_value", "max_installments",
    "n_items", "n_sellers", "max_product_weight_g", "total_product_volume_cm3",
    "n_photos", "purchase_month", "purchase_weekday", "purchase_hour",
]
CATEGORICAL_FEATURES = ["product_category", "payment_type", "customer_state", "seller_state"]

# Recorded so the exclusions are auditable rather than implicit.
EXCLUDED_WITH_REASON = {
    "order_delivered_customer_date": "the target is derived from it",
    "order_estimated_delivery_date": "the target is derived from it",
    "review_score": "written after delivery, so it leaks the outcome",
    "customer_id / seller_id": "identifiers, and high cardinality invites overfitting",
    "purchase_year": "used only to define the temporal holdout in Step 18",
}
for col, reason in EXCLUDED_WITH_REASON.items():
    print(f"excluded  {col:32s} {reason}")

X = pd.get_dummies(df[NUMERIC_FEATURES + CATEGORICAL_FEATURES], columns=CATEGORICAL_FEATURES)
y = df["late"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f"\nfeatures {X.shape[1]} | train {len(X_train):,} ({y_train.mean():.2%} late) | "
      f"test {len(X_test):,} ({y_test.mean():.2%} late)")

excluded  order_delivered_customer_date    the target is derived from it
excluded  order_estimated_delivery_date    the target is derived from it
excluded  review_score                     written after delivery, so it leaks the outcome
excluded  customer_id / seller_id          identifiers, and high cardinality invites overfitting
excluded  purchase_year                    used only to define the temporal holdout in Step 18



features 65 | train 77,043 (8.12% late) | test 19,261 (8.12% late)


## Step 15: Train both models

`scale_pos_weight` and `class_weight="balanced"` re-weight the rare class inside the loss, which is
cleaner than resampling with SMOTE because synthetic oversampling changes the class balance that
precision and recall are measured against.

The support vector machine is fitted on a stratified 15,000-row subsample, since an RBF kernel
scales roughly quadratically and the full training set would take hours against seconds for
gradient boosting. It is still evaluated on the **full test set**, so the comparison stays fair,
but part of any performance gap may reflect training size rather than the algorithm. This is disclosed as a limitation in the report 

In [15]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.08,
    subsample=0.9, colsample_bytree=0.9, scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr", random_state=RANDOM_STATE, n_jobs=-1)
start = time.time()
xgb.fit(X_train, y_train)
print(f"XGBoost fitted on {len(X_train):,} rows in {time.time() - start:.1f}s")

X_svm, _, y_svm, _ = train_test_split(
    X_train, y_train, train_size=SVM_TRAIN_CAP, stratify=y_train, random_state=RANDOM_STATE)
svm = Pipeline([
    ("impute", SimpleImputer(strategy="median")),   # SVMs cannot handle NaN, unlike XGBoost
    ("scale", StandardScaler()),                    # distance-based, so scale matters
    ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced",
                cache_size=1000, random_state=RANDOM_STATE)),
])
start = time.time()
svm.fit(X_svm, y_svm)
print(f"SVM fitted on {len(X_svm):,} rows in {time.time() - start:.1f}s")

XGBoost fitted on 77,043 rows in 1.1s


SVM fitted on 15,000 rows in 3.0s


## Step 16: Evaluate

`decision_function` is used for the SVM rather than `predict_proba`, because `probability=True`
triggers internal five-fold Platt scaling at roughly five times the cost and average precision only
needs a correct ranking, not calibrated probabilities.

In [16]:
xgb_scores = xgb.predict_proba(X_test)[:, 1]
svm_scores = svm.decision_function(X_test)   # signed margin, ranks correctly without calibration

for name, scores in [("XGBoost", xgb_scores), ("SVM", svm_scores)]:
    print(f"{name:8s} PR-AUC {average_precision_score(y_test, scores):.4f}  "
          f"ROC-AUC {roc_auc_score(y_test, scores):.4f}")
print(f"{'random':8s} PR-AUC {y_test.mean():.4f}  (the positive rate)")

XGBoost  PR-AUC 0.4453  ROC-AUC 0.8292
SVM      PR-AUC 0.2677  ROC-AUC 0.7448
random   PR-AUC 0.0812  (the positive rate)


## Step 17: Operating points and the precision-recall trade-off

The two errors cost very different amounts. A missed late delivery produces a support contact, an
unhappy customer and often a credit, whereas a false alarm produces a proactive notification that
costs almost nothing. Recall is therefore worth buying with precision, so the threshold is chosen
by target recall rather than left at the default 0.5.

Precision falls as recall rises because casting a wider net necessarily pulls in more on-time
orders. The right comparison is not against 100% but against the 8.1% base rate, which is what
flagging orders at random would achieve.

In [17]:
def threshold_for_recall(y_true, scores, target):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    ok = np.where(recall[:-1] >= target)[0]          # one more point than thresholds
    return thresholds[ok[np.argmax(precision[:-1][ok])]]

base_rate = y_test.mean()
headline, sweep = [], []

for name, scores in [("XGBoost", xgb_scores), ("SVM", svm_scores)]:
    for target in [0.30, 0.50, 0.70, 0.90]:
        pred = (scores >= threshold_for_recall(y_test, scores, target)).astype(int)
        row = {"model": name, "target_recall": target,
               "precision": precision_score(y_test, pred),
               "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred),
               "lift_vs_base_rate": precision_score(y_test, pred) / base_rate,
               "orders_flagged": int(pred.sum())}
        sweep.append(row)
        if target == 0.70:   # the operating point quoted in the report
            headline.append({"model": f"Olist - {name}",
                             "pr_auc": average_precision_score(y_test, scores),
                             "roc_auc": roc_auc_score(y_test, scores),
                             **{k: row[k] for k in ["precision", "recall", "f1"]}})

pd.DataFrame(headline).to_csv(OUTPUT_DIR / "metrics_olist.csv", index=False)
operating_points = pd.DataFrame(sweep)
operating_points.to_csv(OUTPUT_DIR / "olist_operating_points.csv", index=False)

print(f"base rate (random flagging): {base_rate:.1%} precision\n")
for _, r in operating_points.iterrows():
    print(f"  {r['model']:8s} recall {r['target_recall']:.0%} -> precision {r['precision']:.1%} "
          f"({r['lift_vs_base_rate']:.1f}x base rate), {r['orders_flagged']:,} orders flagged")

base rate (random flagging): 8.1% precision

  XGBoost  recall 30% -> precision 54.3% (6.7x base rate), 869 orders flagged
  XGBoost  recall 50% -> precision 36.7% (4.5x base rate), 2,132 orders flagged
  XGBoost  recall 70% -> precision 24.1% (3.0x base rate), 4,539 orders flagged
  XGBoost  recall 90% -> precision 13.2% (1.6x base rate), 10,704 orders flagged
  SVM      recall 30% -> precision 29.6% (3.6x base rate), 1,597 orders flagged
  SVM      recall 50% -> precision 20.7% (2.6x base rate), 3,776 orders flagged
  SVM      recall 70% -> precision 15.4% (1.9x base rate), 7,115 orders flagged
  SVM      recall 90% -> precision 10.8% (1.3x base rate), 13,004 orders flagged


## Step 18: What drives lateness

Permutation importance is used rather than XGBoost's built-in gain because it is model-agnostic,
which keeps any comparison fair. It measures how far average precision falls when a single column
is shuffled, so a large value means the model genuinely relies on that column. Only the gradient
boosting model is measured, since that is what the report's figure shows.

In [18]:
PERM_SAMPLE = 6000
idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test), PERM_SAMPLE, replace=False)
X_perm, y_perm = X_test.iloc[idx], y_test.iloc[idx]

start = time.time()
result = permutation_importance(
    xgb, X_perm, y_perm, scoring="average_precision",
    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp = (pd.DataFrame({"feature": X_perm.columns, "importance": result.importances_mean})
       .sort_values("importance", ascending=False).reset_index(drop=True))
imp.to_csv(OUTPUT_DIR / "olist_importance_xgb.csv", index=False)

print(f"computed in {time.time() - start:.1f}s\n")
print(imp.head(8).round(4).to_string(index=False))

computed in 3.2s

                feature  importance
window_used_at_handover      0.2468
         purchase_month      0.1116
   promised_window_days      0.0746
            distance_km      0.0510
      customer_state_SP      0.0290
          is_interstate      0.0214
     approval_lag_hours      0.0125
          total_freight      0.0110


## Step 19: Does it hold over time?

The random split mixes 2017 and 2018 orders, which quietly assumes the future resembles the past. A
stricter test trains on 2016 and 2017 and predicts 2018, which is how the model would actually be
deployed. This matters because the late rate is not stable across the period.

In [19]:
print(df.groupby("purchase_year")["late"].agg(["size", "mean"]).round(4).to_string())

is_train_year = (df["purchase_year"] <= 2017).values
X_tr, y_tr = X[is_train_year], y[is_train_year]
X_te, y_te = X[~is_train_year], y[~is_train_year]

xgb_t = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.08, subsample=0.9, colsample_bytree=0.9,
    scale_pos_weight=(y_tr == 0).sum() / (y_tr == 1).sum(),
    eval_metric="aucpr", random_state=RANDOM_STATE, n_jobs=-1).fit(X_tr, y_tr)
X_s, _, y_s, _ = train_test_split(X_tr, y_tr, train_size=min(SVM_TRAIN_CAP, len(X_tr)),
                                  stratify=y_tr, random_state=RANDOM_STATE)
svm_t = Pipeline([
    ("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced", cache_size=1000,
                random_state=RANDOM_STATE))]).fit(X_s, y_s)

temporal = pd.DataFrame([
    {"model": "Olist - XGBoost",
     "pr_auc_random": average_precision_score(y_test, xgb_scores),
     "pr_auc_temporal": average_precision_score(y_te, xgb_t.predict_proba(X_te)[:, 1])},
    {"model": "Olist - SVM",
     "pr_auc_random": average_precision_score(y_test, svm_scores),
     "pr_auc_temporal": average_precision_score(y_te, svm_t.decision_function(X_te))},
])
temporal.to_csv(OUTPUT_DIR / "olist_temporal_check.csv", index=False)
print(f"\n{temporal.round(4).to_string(index=False)}")
print(f"\n2018 late rate {y_te.mean():.2%} against {y_tr.mean():.2%} in training, so a deployed "
      f"model would need regular retraining")

                size    mean
purchase_year               
2016             267  0.0150
2017           43425  0.0663
2018           52612  0.0939



          model  pr_auc_random  pr_auc_temporal
Olist - XGBoost         0.4453           0.3239
    Olist - SVM         0.2677           0.2435

2018 late rate 9.39% against 6.59% in training, so a deployed model would need regular retraining


## Findings

1. Lateness is largely decided **before the parcel moves**: the timing features dominate, pointing
   at the merchant end of the pipeline rather than the courier end.
2. Distance matters far less than intuition suggests, because the promised window is already
   stretched for distant customers.
3. Precision is bounded by how rare late orders are, so the threshold should be set by how many
   proactive interventions the business can afford, not left at a default.
4. The temporal check exposes real drift, so a deployed model would need regular retraining.

## References

**Software**

1. Harris, C. R., Millman, K. J., van der Walt, S. J. et al. 2020. Array programming with NumPy.
   *Nature* 585, 357–362. <https://doi.org/10.1038/s41586-020-2649-2>
2. McKinney, W. 2010. Data Structures for Statistical Computing in Python. In *Proceedings of the
   9th Python in Science Conference*, 56–61. <https://doi.org/10.25080/Majora-92bf1922-00a>
3. Pedregosa, F., Varoquaux, G., Gramfort, A. et al. 2011. Scikit-learn: Machine Learning in
   Python. *Journal of Machine Learning Research* 12, 2825–2830.
   <https://www.jmlr.org/papers/v12/pedregosa11a.html>
4. Chen, T. and Guestrin, C. 2016. XGBoost: A Scalable Tree Boosting System. In *Proceedings of
   the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785–794.
   <https://doi.org/10.1145/2939672.2939785>
5. Kluyver, T., Ragan-Kelley, B., Pérez, F. et al. 2016. Jupyter Notebooks: a publishing format
   for reproducible computational workflows. In *Positioning and Power in Academic Publishing*,
   87–90. <https://doi.org/10.3233/978-1-61499-649-1-87>

**Methods**

6. Cortes, C. and Vapnik, V. 1995. Support-Vector Networks. *Machine Learning* 20, 3, 273–297.
   <https://doi.org/10.1007/BF00994018>
   Used for `SVC`, the algorithm chosen because it was not covered in previous courses.
7. Breiman, L. 2001. Random Forests. *Machine Learning* 45, 1, 5–32.
   <https://doi.org/10.1023/A:1010933404324>
   Origin of permutation importance, used in Step 18 because it is model-agnostic.
8. Saito, T. and Rehmsmeier, M. 2015. The Precision-Recall Plot Is More Informative than the ROC
   Plot When Evaluating Binary Classifiers on Imbalanced Datasets. *PLOS ONE* 10, 3, e0118432.
   <https://doi.org/10.1371/journal.pone.0118432>
   Justifies using average precision rather than accuracy or ROC-AUC at an 8.1% positive rate.

**Data**

9. Olist and Sionek, A. 2018. Brazilian E-Commerce Public Dataset by Olist. Kaggle.
   <https://doi.org/10.34740/kaggle/dsv/195341>
   Accessed 12 August 2026.